In [ ]:
import numpy as np
from sklearn.datasets import fetch_openml
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision.transforms.v2 as transforms_v2
from torchvision import datasets
import matplotlib.pyplot as plt

#CONFIGURATION
SEEDS = [1, 3, 5]
BATCH_SIZE = 100
EPOCHS = 100
LEARNING_RATE = 0.01
MOMENTUM = 0.9
NORMALIZATION = "minmax" #"minmax" or "zscore"
INITIALIZATION = "he" #"he","normal", or "uniform"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CROP_PADDING_NUM = 2
N_FIRST_LAYER = 128
N_SECOND_LAYER = 64
N_THIRD_LAYER = 32
OVERFITTING_DETECTION_PARAMETER = 10

augmenter = transforms_v2.Compose([
    transforms_v2.RandomHorizontalFlip(p=0.5),
    transforms_v2.RandomCrop(28, padding=CROP_PADDING_NUM),
])

train_part = datasets.FashionMNIST(root='./data', train=True, download=True)
test_part = datasets.FashionMNIST(root='./data', train=False, download=True)

X_train_raw = train_part.data
Y_train_raw = train_part.targets
X_test_raw = test_part.data
Y_test_raw = test_part.targets

X = torch.cat([X_train_raw, X_test_raw], dim=0)
Y = torch.cat([Y_train_raw, Y_test_raw], dim=0)


In [ ]:



#DEFINE NN CLASS
class NeuralNetwork(torch.nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = torch.nn.Flatten()
    self.linear_relu_stack = torch.nn.Sequential(
      torch.nn.Linear(28*28,N_FIRST_LAYER),
      torch.nn.ReLU(),
      torch.nn.Linear(N_FIRST_LAYER,N_SECOND_LAYER),
      torch.nn.ReLU(),
      torch.nn.Linear(N_SECOND_LAYER,N_THIRD_LAYER),
      torch.nn.ReLU(),
      torch.nn.Linear(N_THIRD_LAYER,10)
    )

  def forward(self,x):
    x = self.flatten(x)
    logits = self.linear_relu_stack(x)
    return logits

#INITIALIZATION
"""this function checks if the current layer is a linear layer, if it is,
the weights are changed to He init. m represents current layer"""
def weights_init(m):
  if isinstance(m, torch.nn.Linear):
    if(INITIALIZATION == "he"):
      torch.nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
    elif(INITIALIZATION == "normal"):
      torch.nn.init.normal_(m.weight,mean=0,std=0.05)
    else:
      fanin, __ = torch.nn.init._calculate_fan_in_and_fan_out(m.weight)
      torch.nn.init.uniform_(m.weight,a= -1/np.sqrt(fanin),b=1/np.sqrt(fanin))


#TRACK DATA
results = {
    "train_loss": [],
    "valid_loss": [],
    "train_acc": [],
    "valid_acc": [],
    "test_acc": [],
    "converge_time": [],
}

"""this function gets loss and accuracy with a given dataset and labels"""
def eval_model(x_set,y_set,model,criterion):
  with torch.no_grad(): #this disables gradient calculations
    x_set = x_set.to(DEVICE)
    y_set = y_set.to(DEVICE)

    #run through model and get predictions
    outputs = model(x_set)
    loss = criterion(outputs, y_set)
    probab = torch.nn.Softmax(dim=1)(outputs)
    predictions = probab.argmax(1)

    #take off GPU to use numpy
    numpy_pred = predictions.cpu().detach().numpy()
    y_valid_numpy = y_set.cpu().detach().numpy()

    #calc and return the accuracy and loss
    acc = numpy_pred == y_valid_numpy
    return loss.item(),acc.astype(int).sum()/len(acc)




ModuleNotFoundError: No module named 'torch'

In [ ]:

#SIMULATION LOOP
for seed in SEEDS:
  np.random.seed(seed)
  torch.manual_seed(seed)

  #RANDOMLY PERMUTATE DATASET
  indices = np.random.permutation(len(Y))
  X = X[indices]
  Y = Y[indices]

  #SPLIT DATA INTO SPECIFIED REGIONS
  test_set = X[0:7000] #10% of df, 7k data points
  y_test = Y[0:7000]

  valid_set = X[7000:14000] #10% of df, 7k data points
  y_valid = Y[7000:14000]

  train_set = X[14000:] #80% of df, 56k data points
  y_train = Y[14000:]

  #split training again for labeled/unlabeled
  unlabelled_train_set = train_set[0:44800] #80% unlabelled
  y_unlabelled_train = y_train[0:44800]

  labelled_train_set = train_set[44800:] #20% labelled
  y_labelled_train = y_train[44800:]

  #split the labelled training set into actual labeled training or labeled validation
  actual_labelled_train_set = labelled_train_set[0:8400] #15% labeled data for training
  valid_labelled_train_set = labelled_train_set[8400:] #5% for validaiton
  y_actual_labelled_train = y_labelled_train[0:8400]
  y_valid_labelled_train = y_labelled_train[8400:]


  #CONVERT DATA SPLITS TO TENSORS
  test = torch.tensor(test_set, dtype=torch.float32).to(DEVICE)
  y_test = torch.tensor(y_test, dtype=torch.long).to(DEVICE)
  valid = torch.tensor(valid_set, dtype=torch.float32).to(DEVICE)
  y_valid = torch.tensor(y_valid, dtype=torch.long).to(DEVICE)
  train = torch.tensor(train_set, dtype=torch.float32).to(DEVICE)
  y_train = torch.tensor(y_train, dtype=torch.long).to(DEVICE)


  #conv the labeled/unlabeled
  labelled_train_set = torch.tensor(labelled_train_set, dtype=torch.float32).to(DEVICE)#.view(-1, 1, 28, 28)  #since we are doing data augmentation, we have to swap this back to a 2d image. the -1 is placeholder, 1 is one color channel b/c greyscale, 28x28 is 2d image
  unlabelled_train_set = torch.tensor(unlabelled_train_set, dtype=torch.float32).to(DEVICE)
  y_unlabelled_train = torch.tensor(y_unlabelled_train, dtype=torch.long).to(DEVICE)
  y_labelled_train = torch.tensor(y_labelled_train, dtype=torch.long).to(DEVICE)

  valid_labelled_train_set = torch.tensor(valid_labelled_train_set, dtype=torch.float32).to(DEVICE)
  actual_labelled_train_set = torch.tensor(actual_labelled_train_set, dtype=torch.float32).to(DEVICE)
  y_valid_labelled_train = torch.tensor(y_valid_labelled_train,dtype = torch.long).to(DEVICE)
  y_actual_labelled_train = torch.tensor(y_actual_labelled_train,dtype = torch.long).to(DEVICE)



  #INSTANTIATE MODEL
  model = NeuralNetwork().to(DEVICE)
  model.apply(weights_init)

  #DEFINE CRITERIA AND OPTIMIZER
  criterion = torch.nn.CrossEntropyLoss()
  optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE ,momentum = MOMENTUM) #m = 0.5,0.99; lr = 0.001, 0.1
  #optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

  #NORMALIZE VALID AND TEST HERE B/C THEY AREN'T BEING AUGMENTED
  if NORMALIZATION == "minmax":
      test = test / 255.0
      valid = valid / 255.0
      valid_labelled_train_set = valid_labelled_train_set / 255.0
  elif NORMALIZATION == "zscore":
      # Simple Z-score approximation
      valid = (valid - valid.mean()) / valid.std()
      test = (test - test.mean()) / test.std()
      valid_labelled_train_set = (valid_labelled_train_set - valid_labelled_train_set.mean()) / valid_labelled_train_set.std()



  #DEFINE VARS FOR MINIBATCH GRADIENT DESCENT
  dataset = TensorDataset(actual_labelled_train_set, y_actual_labelled_train)
  dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

  #DATA COLLECTION FOR THIS SEED
  seed_train_loss, seed_valid_loss = [], []
  seed_train_acc, seed_valid_acc = [], []
  loss_history = np.zeros(OVERFITTING_DETECTION_PARAMETER)
  loss_history[loss_history == 0.0] = np.nan #fix this later

  #FORWARD PASS WITH INITIALIZED WEIGHTS
  for epoch in range(EPOCHS):
    model.train()

    #VARS FOR MINIBATCH STORAGE
    running_mean_loss = 0 #stores average loss over minibatches, so this is mean per epoch
    running_mean_acc = 0 #stores acc loss over minibatches, so this is acc per epoch
    count = 1 #counts minibatches

    #loads minibatch
    for x,y in dataloader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        x = augmenter(x)
        # plt.imshow(x[0])
        # plt.show()

        if NORMALIZATION == "minmax":
            x = x / 255.0
        elif NORMALIZATION == "zscore":
            x = (x - x.mean()) / x.std()

        #forward pass and loss
        outputs = model(x)
        loss = criterion(outputs, y)

        #backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        #get the loss and acc of the current minibatch
        minibatch_loss, minibatch_acc = eval_model(x, y,model,criterion)

        #store average loss and acc of minibatches
        running_mean_loss = running_mean_loss + (minibatch_loss - running_mean_loss) / count
        running_mean_acc = running_mean_acc + (minibatch_acc - running_mean_acc) / count
        count += 1

    #store the minibatch averages as per-epoch averages
    seed_train_loss.append(running_mean_loss)
    seed_train_acc.append(running_mean_acc)

    #display avg. training loss
    if (epoch + 1) % 10 == 0:
      print(f'Epoch {epoch + 1}, Loss: {running_mean_loss:.4f}')

    #validation
    model.eval()
    valid_loss, valid_acc = eval_model(valid_labelled_train_set, y_valid_labelled_train,model,criterion)

    seed_valid_loss.append(valid_loss)
    seed_valid_acc.append(valid_acc)
    loss_history[epoch%OVERFITTING_DETECTION_PARAMETER] = valid_loss

    #check overfitting
    mean = np.nanmean(loss_history)
    std = np.nanstd(loss_history)
    if valid_loss > mean + std:
      print('overfitting',epoch,valid_loss,valid_acc)
      results['converge_time'].append(epoch)
      break

  results['train_loss'].append(seed_train_loss)
  results['valid_loss'].append(seed_valid_loss)
  results["train_acc"].append(seed_train_acc)
  results["valid_acc"].append(seed_valid_acc)


  #TESTING
  logits = model(test)
  pred_probab = torch.nn.Softmax(dim=1)(logits)
  y_pred = pred_probab.argmax(1)
  print(f"Predicted class: {y_pred}")

  _,final_acc = eval_model(test,y_test,model,criterion)

  results["test_acc"].append(final_acc)


In [ ]:
def pad_results(results,convergence_list):
  max_ = max(convergence_list) +1 #this is the value we extend the other lists to
  

  for i in range(len(results["train_loss"])):
    get_max = results['converge_time'][i]+1
    #if the current seed isnt the one with the longest iterations, pad it to reach the max iterations

    while get_max < max_:
      results["valid_acc"][i].append(results["valid_acc"][i][-1])
      results["train_loss"][i].append(results["train_loss"][i][-1])
      results["valid_loss"][i].append(results["valid_loss"][i][-1])
      results["train_acc"][i].append(results["train_acc"][i][-1])
      get_max+=1

pad_results(results,results["converge_time"])

NameError: name 'model' is not defined

In [ ]:

#create dataframe

data_dict = {'training_loss': np.mean(np.array(results["train_loss"]),axis=0), 'validation_loss': np.mean(np.array(results["valid_loss"]),axis=0), 'training_accuracy': np.mean(np.array(results["train_acc"]),axis=0), 'validation_accuracy': np.mean(np.array(results["valid_acc"]),axis=0)}
pd.DataFrame(data_dict)
pd.DataFrame(data_dict).plot()



In [ ]:
np.mean(results['test_acc'])

In [ ]:
np.mean(results['converge_time'])